In [1]:
import zipfile

with zipfile.ZipFile('brotherhood.zip') as zip_ref:
    zip_ref.extractall()

In [2]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('train_data.csv')
test_df = pd.read_csv('test_data.csv')

queries = test_df[test_df['type'] == 'query'].reset_index(drop=True)
candidates = test_df[test_df['type'] == 'candidate'].reset_index(drop=True)
train_df.head(3)

,pair_id,py_source,cpp_source
0,pair_0000,"def f_gold ( num , divisor ) :\n while ( nu...","using namespace std;\nint f_gold ( int num, in..."
1,pair_0001,"def f_gold ( arr , n ) :\n mp = dict ( )\n ...",using namespace std;\nint f_gold ( int arr [ ]...
2,pair_0002,def f_gold ( s ) :\n length = len ( s )\n ...,using namespace std;\nint f_gold ( string str ...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=20000)
vectorizer.fit(test_df['source'].tolist())

query_vecs = vectorizer.transform(queries['source'])
candidate_vecs = vectorizer.transform(candidates['source'])

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

sims = cosine_similarity(query_vecs, candidate_vecs)
print(f'Similarity matrix shape: {sims.shape}')
print(f'Query 0 top match: candidate idx {sims[0].argmax()}, sim={sims[0].max():.4f}')

Similarity matrix shape: (200, 200)
Query 0 top match: candidate idx 55, sim=0.8344


In [5]:
rows = []
for i, row in queries.iterrows():
    ranked_indices = np.argsort(-sims[i])
    ranked_ids = ';'.join(candidates.iloc[ranked_indices]['datapointID'].values)
    rows.append({'subtaskID': 1, 'datapointID': row['datapointID'], 'answer': ranked_ids})

output_df = pd.DataFrame(rows)
output_df.to_csv('subi.csv', index=False)
output_df.head()

,subtaskID,datapointID,answer
0,1,py_test_public_0000,candidate_0055;candidate_0144;candidate_0084;c...
1,1,py_test_public_0001,candidate_0167;candidate_0038;candidate_0177;c...
2,1,py_test_public_0002,candidate_0007;candidate_0008;candidate_0101;c...
3,1,py_test_public_0003,candidate_0019;candidate_0100;candidate_0179;c...
4,1,py_test_public_0004,candidate_0101;candidate_0008;candidate_0007;c...
